# 02 — QA and Baselines

Verification checks (`qa.py`) against whatever processed artifacts already exist, plus the PCA dimension search (manifest E7).

In [ ]:
import sys
sys.path.insert(0, '..')

from src.amr_data.config import load_paths
from src.amr_data.io import load_pickle
from src.amr_data import qa

paths = load_paths()

## Gene presence/absence QA (B4/B5/B6/B9)

In [ ]:
gf = load_pickle(paths.processed('gene_filtered_legacy_pkl'))
pa = load_pickle(paths.processed('gene_pa_pkl'))
report = qa.verify_gene_pa(pa, gf, genome_names_reference=gf['genome_names'])
report

In [ ]:
qa.export_unmatched_strains(pa, paths.results_dir / 'qa' / 'unmatched_strains.csv')

## PCA dimension search (reproduces manifest E7)

In [ ]:
from src.amr_data.baselines import pca_dim_search

snp_data = load_pickle(paths.processed('snp_matrix_pkl'))
master = load_pickle(paths.processed('master_features_pkl'))
rs = master['random_split']

results = pca_dim_search(
    snp_data['X_snp'], rs['train_idx'], rs['val_idx'], rs['test_idx'],
    master['Y'].astype('float64'), master['label_mask'], list(master['label_names']),
)
for dim, r in results.items():
    print(f"dim={dim}: var={r['variance_retained']:.3f}  avg_pr_auc={r['avg_pr_auc']:.4f}")